# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hafiz-Taha-Hussain/Flyrank-Work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip install -q duckdb huggingface_hub

import duckdb
from huggingface_hub import login, HfApi

# In Colab: key icon on the left -> add a secret named HF_TOKEN -> toggle "notebook access" on.
# Never paste the token itself into a cell — this repo is public.
from google.colab import userdata
HF_TOKEN = userdata.get("HF-TOKEN")
login(token=HF_TOKEN)

REPO_ID = "FlyRank/internship-warehouse"
api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")

# Show the fact table's files so we can see the real partition naming
# (e.g. whether it's month=2026-03 style Hive partitioning) before writing any query.
fact_files = [f for f in files if "fact_content_daily_performance" in f and "sample" not in f]
print(f"{len(fact_files)} fact table files found. First 15:")
for f in fact_files[:15]:
    print(" ", f)

18 fact table files found. First 15:
  fact_content_daily_performance/month=2025-01/data_0.parquet
  fact_content_daily_performance/month=2025-02/data_0.parquet
  fact_content_daily_performance/month=2025-03/data_0.parquet
  fact_content_daily_performance/month=2025-04/data_0.parquet
  fact_content_daily_performance/month=2025-05/data_0.parquet
  fact_content_daily_performance/month=2025-06/data_0.parquet
  fact_content_daily_performance/month=2025-07/data_0.parquet
  fact_content_daily_performance/month=2025-08/data_0.parquet
  fact_content_daily_performance/month=2025-09/data_0.parquet
  fact_content_daily_performance/month=2025-10/data_0.parquet
  fact_content_daily_performance/month=2025-11/data_0.parquet
  fact_content_daily_performance/month=2025-12/data_0.parquet
  fact_content_daily_performance/month=2026-01/data_0.parquet
  fact_content_daily_performance/month=2026-02/data_0.parquet
  fact_content_daily_performance/month=2026-03/data_0.parquet


In [8]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');""")

# Adjust this glob to match whatever partition pattern the file listing above actually shows.
MARCH_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-03/*.parquet"

# Quick smoke test: does this path resolve to anything at all?
con.sql(f"SELECT COUNT(*) AS row_count FROM '{MARCH_GLOB}'").df()

,row_count
0,9841378


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row of `fact_content_daily_performance`** = one content page (`content_hash_id`), for one
client (`client_hash_id`), on one calendar day (`report_date`) — a page-day grain, not a
page-level summary. That's a finer grain than the w01/w02 starter CSV, which was already
pre-aggregated to one row per page over a trailing 90 days.

**Time window:** `month=2026-03` — a mid-panel month, per the warning that the final month
(June 2026, which is also all `_sample` contains) is the natural outcome window for any
past→future label and must stay a sealed test month, not something I develop label logic
against.

**Table(s):** the full `fact_content_daily_performance` table, filtered to the March
partition (NOT `fact_content_daily_performance_sample`, since the sample is exactly June —
using it here would silently mean "iterating" on the same month I'm supposed to hold out
later). `dim_content` and `dim_clients` for context/joins where needed.

In [4]:
# Verify the window really is March, and see the raw row count before any filtering.
con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*)         AS raw_row_count
FROM '{MARCH_GLOB}'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,raw_row_count
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I ran `DESCRIBE` below to see the real column names before sorting them — a bucket list
written from memory instead of the actual schema is exactly the kind of "guess, not a
contract" the skill warns about. Buckets, using the columns confirmed by the query below:

- **Context (never a feature — grouping/joining/reading only):** `client_hash_id`,
  `content_hash_id`, `report_date`, `month` — the grain itself, plus `client_has_gsc`/
  `client_has_ga4` (client-level flags used to filter, not to learn from).
- **Feature (knowable before the decision moment, safe to use):** prior-window
  aggregates built only from March — `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`
  (via `gsc_sum_position`), `ga4_pageviews`, `ga4_sessions`, `ga4_engaged_sessions`,
  `ga4_total_engagement_sec`, the channel breakdown (`sessions_organic`, `sessions_direct`,
  `sessions_referral`, `sessions_social`, `sessions_paid`), the AI-referrer breakdown
  (`sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`,
  `ai_other`), and `scroll_events` — all knowable because they only reflect days that have
  already happened by the review moment.
- **Label / proxy (the thing being predicted, never a feature):** a forward-looking
  decline signal built from a *later* month than the features — e.g. whether
  `gsc_clicks` drops from March to April. This is the fix for the w01/w02 proxy problem
  (`trend_direction` there was circular, computed from the same window it labeled); here
  the label genuinely comes from a future window relative to the features.
- **Excluded:** `gsc_data_available`/`ga4_data_available` themselves are context/filters,
  never features — a flag saying "do we have data" isn't a signal about page performance.
  Any `ga4_*` column is excluded from feature-building for rows where
  `ga4_data_available IS NOT TRUE` (per the panel warning, those are zero-filled, not
  really zero). The `_sample` table is excluded entirely from label-building (June-only,
  would leak the eventual sealed test month).

In [9]:
# Real schema check — confirms which of the column names above actually exist,
# and surfaces any I guessed wrong so the bucket list above can be corrected.
con.sql(f"DESCRIBE SELECT * FROM '{MARCH_GLOB}' LIMIT 0").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3a. Grain check
Proves one row really is one (client, content, day) — zero rows back means the grain holds.

In [10]:
con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
FROM '{MARCH_GLOB}'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,c


### 3b. Row count and date span for this slice
Confirms the real volume for March and that the dates genuinely stay inside the month.

In [11]:
con.sql(f"""
SELECT
    COUNT(*)                    AS row_count,
    COUNT(DISTINCT client_hash_id)   AS n_clients,
    COUNT(DISTINCT content_hash_id)  AS n_content_items,
    MIN(report_date)            AS min_date,
    MAX(report_date)            AS max_date
FROM '{MARCH_GLOB}'
""").df()

,row_count,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### 3c. Availability — filter with `IS TRUE`
Shows how many rows survive once the GA4 availability flag is actually respected, versus
the raw count above. The gap is exactly the rows that would otherwise look like "zero
engagement" but are really "no data yet for this client."

In [12]:
result = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4_available
FROM '{MARCH_GLOB}'
""").df()
result["pct_available"] = (result["rows_with_ga4_available"] / result["total_rows"] * 100).round(1)
result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4_available,pct_available
0,9841378,413966.0,4.2


### 3d. Five features, max — each with an "available when?" line

Built from March data only, at the (client, content) grain — one row per page for the
month, aggregating only over days at-or-before the point I'd be scoring it.

In [13]:
features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)  AS avg_position_march,
    SUM(gsc_impressions)                                            AS impressions_march,
    SUM(gsc_clicks)                                                 AS clicks_march,
    SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_march,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_engaged_sessions ELSE 0 END)                  AS engaged_sessions_march
FROM '{MARCH_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

print(features.shape)
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 7)


,client_hash_id,content_hash_id,avg_position_march,impressions_march,clicks_march,ctr_march,engaged_sessions_march
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6.893301,6523.0,7.0,0.001073,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,3.214128,453.0,0.0,0.000000,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,6.535346,5630.0,6.0,0.001066,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,7.435680,4944.0,13.0,0.002629,0.0
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,15.785714,42.0,0.0,0.000000,0.0
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,3.871795,429.0,1.0,0.002331,0.0
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,10.538117,223.0,1.0,0.004484,0.0
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,5.864583,96.0,1.0,0.010417,0.0
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,10.057325,314.0,1.0,0.003185,0.0
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,5.127643,7709.0,20.0,0.002594,0.0


**Available when? — one line per feature:**

1. `avg_position_march` — knowable at the decision moment because it's an impressions-weighted average (`SUM(gsc_sum_position)/SUM(gsc_impressions)`) of *already-occurred* daily rankings within March, never a future ranking. (Weighted this way instead of a naive AVG-of-daily-averages, since days with more impressions should count more.)
2. `impressions_march` — a running sum of search impressions that already happened by the
   time the page is reviewed; no future days included.
3. `clicks_march` — same reasoning: a sum over days that have already happened.
4. `ctr_march` — derived purely from the two features above, so it inherits their
   availability; no leakage introduced by the ratio itself.
5. `engaged_sessions_march` — restricted to rows where `ga4_data_available IS TRUE`, so it
   never silently counts a zero-filled placeholder as a real (lack of) engagement signal.

### 3e. The trap — a deliberate leak

Step 1: an honest quick score using only the five clean features above, predicting a
forward-looking decline label (built from the *next* month, not from March itself — this
matters, see the label bucket in section 2). Step 2: add one column that's derived from the
same window as the label on purpose, watch the score jump. Step 3: delete it, keep the
honest number.

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Forward-looking label: built from April (the month AFTER the feature window),
# never from March itself — this is the fix for the circular w01/w02 proxy.
APRIL_GLOB = f"hf://datasets/{REPO_ID}/**/month=2026-04/*.parquet"

label = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) AS clicks_april
FROM '{APRIL_GLOB}'
GROUP BY client_hash_id, content_hash_id
""").df()

data = features.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
data["declined_in_april"] = (data["clicks_april"] < data["clicks_march"]).astype(int)

clean_feature_cols = [
    "avg_position_march", "impressions_march", "clicks_march",
    "ctr_march", "engaged_sessions_march",
]

X = data[clean_feature_cols].fillna(0)
y = data["declined_in_april"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (5 clean features only): {honest_auc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest AUC (5 clean features only): 0.944


In [15]:
# --- THE TRAP: add one column derived from the SAME window as the label ---
# clicks_april is literally what the label is computed from — this should never be a feature.
leaky_feature_cols = clean_feature_cols + ["clicks_april"]

X_leak = data[leaky_feature_cols].fillna(0)
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_leak, y, test_size=0.3, random_state=42
)

leaky_model = LogisticRegression(max_iter=1000).fit(X_train_leak, y_train_leak)
leaky_auc = roc_auc_score(y_test_leak, leaky_model.predict_proba(X_test_leak)[:, 1])
print(f"Leaky AUC (clicks_april smuggled in as a feature): {leaky_auc:.3f}")
print(f"Jump from honest to leaky: {leaky_auc - honest_auc:+.3f}")

Leaky AUC (clicks_april smuggled in as a feature): 1.000
Jump from honest to leaky: +0.056


In [16]:
# Delete the leaked column and keep only the honest number.
del data["clicks_april"]
print(f"Leaked column removed. Keeping the honest score: {honest_auc:.3f}")
print("The gap above is the leakage lesson from notebook 02, reproduced on real warehouse data:")
print("a feature drawn from the same window as the label doesn't predict the future — it just restates it.")

Leaked column removed. Keeping the honest score: 0.944
The gap above is the leakage lesson from notebook 02, reproduced on real warehouse data:
a feature drawn from the same window as the label doesn't predict the future — it just restates it.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced client history.** Per the warehouse docs, `dim_clients.gsc_data_start` varies
  a lot per client — a global March window doesn't mean every client has a full month of
  real history; some rows in this slice may sit right at the edge of a client's earliest
  data. I did not check per-client start dates in this pass, so counts by client should be
  treated as an approximation until that's verified.
- **GA4 availability isn't uniform.** Section 3c shows the real gap between total rows and
  rows with genuine GA4 data — for the portion excluded, engagement-based features simply
  don't exist yet, they aren't zero.
- **This data cannot establish causality.** It can say a page's signals correlate with a
  later click drop; it cannot say refreshing the page would have prevented it.
- **A third of clients have little or no usable search/analytics history** per the warehouse
  docs — this slice's five features will be systematically weaker or missing for those
  clients, not randomly missing.
- **The `_sample` table stays untouched for label logic** in this notebook on purpose — it's
  June only, the eventual sealed test month, so nothing here was iterated against it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.